# Kenya Maize Advisory Chatbot

## Local PDF RAG for English & Swahili

Answers questions about **maize farming in Kenya** by retrieving evidence from the documents in `pdfs/` plus the project's own `DISEASE_INFO` seed knowledge, then generating a grounded, source-cited reply.

```
PDFs --> extract --> clean --> language tag --> chunk
      --> (section 3) embed --> hybrid retrieval --> guardrails --> generate --> translate --> cite
        DISEASE_INFO (4 leaf diseases) feeds in at the chunking step
```

- **Languages:** English and Swahili, using a translation pivot (Swahili query -> English retrieval/generation -> Swahili answer).
- **Stack:** everything free and local - `pymupdf`, `multilingual-e5-small`, `numpy`, `rank_bm25`, `nllb-200`, Qwen2.5. No API keys.
- **Artifacts:** `rag_data/` (corpus, chunks), `rag_artifacts/` (index, config, evaluation report), MySQL chat history (section 10).

Sections 1-2 below build the environment and the corpus. Later sections embed, retrieve, evaluate, tune and deploy.

## 1. Environment & Setup

### 1.1 Project objectives

- Build a knowledge base from the PDFs in `pdfs/` plus `DISEASE_INFO`, covering Kenyan maize farming: production and agronomy, seed varieties by agro-ecological zone, soil fertility and fertiliser, pests (including fall armyworm) and diseases, post-harvest handling and storage, farm economics and policy.
- Retrieve answers to farmer questions in **English or Swahili** and generate replies that stay inside the retrieved evidence, with citations back to source page numbers.
- Export a reusable pipeline (`rag_pipeline.py`) that the Streamlit app loads as a **Maize Advisory** chat tab, continuing the conversation after an image classification.

Everything runs on the CPU, locally, for free.

### 1.2 Dependency installation

This lists which packages are installed (pymupdf, torch, scipy, etc.) and stops if the essential ones are missing.

In [83]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    %pip install -q --force-reinstall --no-deps sentence-transformers rank-bm25 pymupdf scipy scikit-learn

import importlib.util
import pandas as pd

REQUIRED = {
    "pymupdf": "PDF text extraction (section 2)",
    "pandas": "audit tables (section 2)",
    "torch": "embedding model runtime (section 3)",
    "sentence_transformers": "embeddings (section 3)",
    "rank_bm25": "hybrid search (section 4)",
    "scipy": "numerics behind sentence-transformers (all sections)",
    "sklearn": "used by sentence-transformers (all sections)",
}
availability = [
    {"package": name, "installed": importlib.util.find_spec(name) is not None, "needed for": purpose}
    for name, purpose in REQUIRED.items()
]
print(pd.DataFrame(availability).to_string(index=False))
missing = [name for name, row in zip(REQUIRED, availability) if not row["installed"]]
blocking_now = [name for name in missing if name in ("pymupdf", "pandas")]
if blocking_now:
    raise ImportError("Missing required packages: " + ", ".join(blocking_now) + ". Set INSTALL_DEPENDENCIES=True and rerun.")
if missing:
    print("\nNot installed yet (needed from section 3):", ", ".join(missing))

              package  installed                                           needed for
              pymupdf       True                      PDF text extraction (section 2)
               pandas       True                             audit tables (section 2)
                torch       True                  embedding model runtime (section 3)
sentence_transformers       True                               embeddings (section 3)
            rank_bm25       True                            hybrid search (section 4)
                scipy       True numerics behind sentence-transformers (all sections)
              sklearn       True         used by sentence-transformers (all sections)


### 1.3 Imports, device & seeding

In [84]:
import ast
import hashlib
import json
import random
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"python stack ready | device={DEVICE} | seed={SEED}")
print(f"pymupdf {pymupdf.__version__} | numpy {np.__version__} | pandas {pd.__version__} | torch {torch.__version__}")

python stack ready | device=mps | seed=42
pymupdf 1.27.1 | numpy 2.5.1 | pandas 2.3.3 | torch 2.14.0


### 1.4 RAG configuration

This sets folder paths, the embedding model, chunk size (220 words), overlap (40 words) and top-K (5). It checks that app.py and pdfs/ exist.

In [85]:
ROOT         = Path.cwd()      
PDF_DIR      = ROOT / "pdfs"
DATA_DIR     = ROOT / "rag_data"
ARTIFACT_DIR = ROOT / "rag_artifacts"

EMBEDDING_MODEL = "intfloat/multilingual-e5-small"  
LANGUAGES       = ("en", "sw")
CHUNK_WORDS     = 220
OVERLAP_WORDS   = 40
MIN_CHUNK_WORDS = 30
TOP_K           = 5

if not (ROOT / "app.py").is_file() or not PDF_DIR.is_dir():
    raise FileNotFoundError(
        f"Run this notebook from the repository root (expected app.py and pdfs/ under {ROOT})."
    )
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

print("RAG configuration")
print(f"  root            : {ROOT}")
print(f"  pdfs            : {PDF_DIR} ({len(list(PDF_DIR.glob('*.pdf')))} PDFs)")
print(f"  data dir        : {DATA_DIR}")
print(f"  embedding model : {EMBEDDING_MODEL}  (section 3)")
print(f"  chunking        : {CHUNK_WORDS} words, {OVERLAP_WORDS} overlap, min {MIN_CHUNK_WORDS}")
print(f"  top_k           : {TOP_K}")
print(f"  languages       : {', '.join(LANGUAGES)}")

RAG configuration
  root            : /Users/willard/school/maize_disease
  pdfs            : /Users/willard/school/maize_disease/pdfs (24 PDFs)
  data dir        : /Users/willard/school/maize_disease/rag_data
  embedding model : intfloat/multilingual-e5-small  (section 3)
  chunking        : 220 words, 40 overlap, min 30
  top_k           : 5
  languages       : en, sw


### 1.5 Reuse existing assets (`DISEASE_INFO`)

The Streamlit app holds curated disease knowledge in a `DISEASE_INFO` dict. The four disease entries become seed documents so the chatbot can answer symptom and management questions even before a PDF covers them.

`app.py` executes Streamlit code at import time, so the dict is read with the `ast` module instead of importing the file.

In [86]:
app_source = (ROOT / "app.py").read_text(encoding="utf-8")
tree = ast.parse(app_source)

DISEASE_INFO = None
for node in tree.body:
    target = node.target.id if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) else None
    if target is None and isinstance(node, ast.Assign):
        target = node.targets[0].id if isinstance(node.targets[0], ast.Name) else None
    if target == "DISEASE_INFO":
        DISEASE_INFO = ast.literal_eval(node.value)
        break

if DISEASE_INFO is None:
    raise ValueError("DISEASE_INFO not found in app.py")

EXPECTED_CLASSES = ["Blight", "Common_Rust", "Gray_Leaf_Spot", "Healthy"]
if set(DISEASE_INFO) != set(EXPECTED_CLASSES):
    raise ValueError(f"Unexpected classes in DISEASE_INFO: {sorted(DISEASE_INFO)}")

for cls, info in DISEASE_INFO.items():
    missing_fields = {"pathogen", "symptoms", "favourable", "management"} - set(info)
    if missing_fields:
        raise ValueError(f"{cls} is missing fields: {sorted(missing_fields)}")

print(f"loaded DISEASE_INFO: {len(DISEASE_INFO)} diseases x {len(next(iter(DISEASE_INFO.values())))} fields")
for cls, info in DISEASE_INFO.items():
    print(f"  {cls:15s} {info['pathogen']}")

loaded DISEASE_INFO: 4 diseases x 4 fields
  Blight          Exserohilum turcicum (formerly Bipolaris turcica)
  Common_Rust     Puccinia sorghi
  Gray_Leaf_Spot  Cercospora zeae-maydis
  Healthy         -


## 2. Data Ingestion & Chunking

### 2.1 Source manifest & licences

Roles below are **inferred from filenames** - read each PDF and correct the `title` / `topic` fields if the content does not match. `licence` is deliberately `unverified`: check each document's licence before redistributing the corpus, and record what you find in this table (it is saved into `source_manifest.json` in section 2.8).

| source_id | PDF | Inferred role | Topic |
|---|---|---|---|
| S01 | diversity in maize production environments and practices | production environments & on-farm practices | agronomy |
| S02 | MAIZE GROWERS GUIDE | practical growers handbook | agronomy |
| S03 | National-Agriculture-Production-Report-2025 | national production statistics | statistics |
| S04 | Olwande_Smallholder_Maize_Efficiency_Kenya | smallholder maize efficiency study | economics |
| S05 | Planting_strategies_of_maize_farmers_in_Kenya_a_si | research article on planting strategies | agronomy |
| S06 | CIMMYT_Maize_Diseases_Field_Identification_4thEd | field guide to maize disease identification | disease |
| S07 | CIMMYT_MLN_Technical_Manual_Disease_Management | MLN disease management technical manual | disease |
| S08 | KALRO_MLND_Kenya_Training_Manual | Kenya MLND extension training manual | disease |
| S09 | KEPHIS_National_Crop_Variety_List_2025 | official maize variety register (altitude, maturity, yield, tolerance) | seed |
| S10 | MoA_Sustainable_Agricultural_Livelihood_Restoration_Training_Manual | conservation agriculture, land prep, ISFM, soils | agronomy |
| S11 | KALRO_Soil_Fertility_Soil_Testing_Overview | soil fertility constraints & fertilizer recommendations | fertilizer |
| S12 | IFDC_Kenya_Central_Fertilizer_Use_Optimizer_DAP_CAN | per-acre DAP/CAN fertilizer rates | fertilizer |
| S13 | IFDC_Kenya_Lower_Western_Fertilizer_Use_Optimizer_2023 | per-acre DAP/Urea fertilizer rates | fertilizer |
| S14 | CABI_Fall_Armyworm_Technical_Brief_Kenya_2018 | fall armyworm ID, scouting and control | pest |
| S15 | CIMMYT_Fall_Armyworm_IPM_Strategic_Plan_Kenya | fall armyworm integrated pest management plan | pest |
| S16 | FAO_Integrated_Management_Fall_Armyworm_Maize_FFS | fall armyworm management for farmer field schools | pest |
| S17 | IITA_Aflatoxin_Management_Maize_Groundnuts_Kenya | aflatoxin prevention incl. Aflasafe KE01 | postharvest |
| S18 | KALRO_Aflasafe_KE01_Brief | Aflasafe KE01 biocontrol brief | postharvest |
| S19 | FAO_Maize_Post_Harvest_Compendium | maize drying, moisture and storage | postharvest |
| S20 | TAAT_PICS_Hermetic_Bags_Grain_Storage_Guide | hermetic (PICS) bag storage | postharvest |
| S21 | CIMMYT_Maize_Storage_Pest_Control_Metal_Silo | metal silo storage pest control | postharvest |
| S22 | FAO_MAFAP_Kenya_Maize_Policy_Note | maize pricing and input subsidy policy | policy |
| S23 | NCPB_Act_Cap338 | NCPB grading, pricing and buying regulations | policy |
| S24 | NCPB_Restructuring_Report_2014_Cost_Of_Production | cost of maize production breakdown | policy |
| S00 | `DISEASE_INFO` from `app.py` | leaf-disease seed knowledge | disease |

In [87]:
SOURCES = [
    {"source_id": "S00", "filename": None, "title": "Maize leaf disease seed knowledge (app.py)",
     "topic": "disease", "language": "en", "licence": "project asset"},
    {"source_id": "S01", "filename": "diversity in maize production environments and practices.pdf",
     "title": "Diversity in maize production environments and practices", "topic": "agronomy",
     "language": "unknown", "licence": "unverified"},
    {"source_id": "S02", "filename": "MAIZE GROWERS GUIDE.pdf",
     "title": "Maize Growers Guide", "topic": "agronomy", "language": "unknown", "licence": "unverified"},
    {"source_id": "S03", "filename": "National-Agriculture-Production-Report-2025.pdf",
     "title": "National Agriculture Production Report 2025", "topic": "statistics",
     "language": "unknown", "licence": "unverified"},
    {"source_id": "S04", "filename": "Olwande_Smallholder_Maize_Efficiency_Kenya.pdf",
     "title": "Smallholder maize efficiency in Kenya (Olwande)", "topic": "economics",
     "language": "unknown", "licence": "unverified"},
    {"source_id": "S05", "filename": "Planting_strategies_of_maize_farmers_in_Kenya_a_si.pdf",
     "title": "Planting strategies of maize farmers in Kenya", "topic": "agronomy",
     "language": "unknown", "licence": "unverified"},
    {"source_id": "S06", "filename": "CIMMYT_Maize_Diseases_Field_Identification_4thEd.pdf",
     "title": "Maize Diseases: A Guide for Field Identification (4th ed., CIMMYT)", "topic": "disease",
     "language": "en", "licence": "free to share (CIMMYT Intl Public Good)"},
    {"source_id": "S07", "filename": "CIMMYT_MLN_Technical_Manual_Disease_Management.pdf",
     "title": "Maize Lethal Necrosis (MLN): A Technical Manual for Disease Management (CIMMYT)", "topic": "disease",
     "language": "en", "licence": "CIMMYT publication"},
    {"source_id": "S08", "filename": "KALRO_MLND_Kenya_Training_Manual.pdf",
     "title": "Understanding & Managing Maize Lethal Necrosis Disease (MLND) in Kenya", "topic": "disease",
     "language": "en", "licence": "Government of Kenya / KALRO"},
    {"source_id": "S09", "filename": "KEPHIS_National_Crop_Variety_List_2025.pdf",
     "title": "National Crop Variety List - Kenya (2025 edition)", "topic": "seed",
     "language": "en", "licence": "KEPHIS / Government of Kenya"},
    {"source_id": "S10", "filename": "MoA_Sustainable_Agricultural_Livelihood_Restoration_Training_Manual.pdf",
     "title": "Sustainable Agricultural Livelihood Restoration Training Manual (MoA)", "topic": "agronomy",
     "language": "en", "licence": "Government of Kenya"},
    {"source_id": "S11", "filename": "KALRO_Soil_Fertility_Soil_Testing_Overview.pdf",
     "title": "Overview of soil fertility & soil-testing programs for fertilizer recommendations in Kenya (KALRO)", "topic": "fertilizer",
     "language": "en", "licence": "KALRO / IFDC"},
    {"source_id": "S12", "filename": "IFDC_Kenya_Central_Fertilizer_Use_Optimizer_DAP_CAN.pdf",
     "title": "Kenya Central Fertilizer Use Optimizer (DAP-CAN)", "topic": "fertilizer",
     "language": "en", "licence": "IFDC"},
    {"source_id": "S13", "filename": "IFDC_Kenya_Lower_Western_Fertilizer_Use_Optimizer_2023.pdf",
     "title": "Kenya Lower Western Fertilizer Use Optimizer (DAP-Urea, 2023)", "topic": "fertilizer",
     "language": "en", "licence": "IFDC"},
    {"source_id": "S14", "filename": "CABI_Fall_Armyworm_Technical_Brief_Kenya_2018.pdf",
     "title": "Fall Armyworm Technical Brief (Maize, Kenya 2018)", "topic": "pest",
     "language": "en", "licence": "CABI / Ministry of Agriculture"},
    {"source_id": "S15", "filename": "CIMMYT_Fall_Armyworm_IPM_Strategic_Plan_Kenya.pdf",
     "title": "Fall Armyworm for Maize Smallholders in Kenya: An IPM Strategic Plan (CIMMYT)", "topic": "pest",
     "language": "en", "licence": "CIMMYT publication"},
    {"source_id": "S16", "filename": "FAO_Integrated_Management_Fall_Armyworm_Maize_FFS.pdf",
     "title": "Integrated Management of the Fall Armyworm on Maize: A Guide for Farmer Field Schools in Africa (FAO)", "topic": "pest",
     "language": "en", "licence": "FAO"},
    {"source_id": "S17", "filename": "IITA_Aflatoxin_Management_Maize_Groundnuts_Kenya.pdf",
     "title": "Management of Aflatoxins in Maize and Groundnuts in Kenya: A Farmers' Training Manual (IITA)", "topic": "postharvest",
     "language": "en", "licence": "IITA / CGIAR"},
    {"source_id": "S18", "filename": "KALRO_Aflasafe_KE01_Brief.pdf",
     "title": "Aflasafe KE01 Brief (KALRO-Katumani)", "topic": "postharvest",
     "language": "en", "licence": "KALRO"},
    {"source_id": "S19", "filename": "FAO_Maize_Post_Harvest_Compendium.pdf",
     "title": "Maize Post-harvest Operations Compendium (FAO)", "topic": "postharvest",
     "language": "en", "licence": "FAO"},
    {"source_id": "S20", "filename": "TAAT_PICS_Hermetic_Bags_Grain_Storage_Guide.pdf",
     "title": "A Guide on the Use of PICS Bags for Grain Storage (TAAT)", "topic": "postharvest",
     "language": "en", "licence": "TAAT / Purdue"},
    {"source_id": "S21", "filename": "CIMMYT_Maize_Storage_Pest_Control_Metal_Silo.pdf",
     "title": "Maize Storage Pest Control in Sub-Saharan Africa: Metal Silo Technology (CIMMYT)", "topic": "postharvest",
     "language": "en", "licence": "CIMMYT publication"},
    {"source_id": "S22", "filename": "FAO_MAFAP_Kenya_Maize_Policy_Note.pdf",
     "title": "Kenya maize policy note (FAO MAFAP)", "topic": "policy",
     "language": "en", "licence": "FAO"},
    {"source_id": "S23", "filename": "NCPB_Act_Cap338.pdf",
     "title": "National Cereals and Produce Board Act (Cap 338)", "topic": "policy",
     "language": "en", "licence": "Kenya Law"},
    {"source_id": "S24", "filename": "NCPB_Restructuring_Report_2014_Cost_Of_Production.pdf",
     "title": "NCPB Restructuring Report 2014 (cost of maize production)", "topic": "policy",
     "language": "en", "licence": "NCPB"},
]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

PDF_SOURCES = [s for s in SOURCES if s["filename"]]
missing = [s["filename"] for s in PDF_SOURCES if not (PDF_DIR / s["filename"]).is_file()]
if missing:
    raise FileNotFoundError("Put these PDFs in " + str(PDF_DIR) + ":\n" + "\n".join(missing))

SOURCE_HASHES = {}
inventory = []
for src in PDF_SOURCES:
    path = PDF_DIR / src["filename"]
    SOURCE_HASHES[src["source_id"]] = sha256_file(path)
    inventory.append({
        "source_id": src["source_id"], "filename": src["filename"], "topic": src["topic"],
        "size_mb": round(path.stat().st_size / 1e6, 2), "sha256": SOURCE_HASHES[src["source_id"]][:12],
        "licence": src["licence"],
    })

hashes = list(SOURCE_HASHES.values())
if len(set(hashes)) != len(hashes):
    raise ValueError("Two PDFs have identical content - remove the duplicate before chunking.")

display(pd.DataFrame(inventory))
print(f"{len(PDF_SOURCES)} PDFs inventoried, all unique, ready to extract.")

,source_id,filename,topic,size_mb,sha256,licence
0,S01,diversity in maize production environments and...,agronomy,0.67,1aabed3d8f7f,unverified
1,S02,MAIZE GROWERS GUIDE.pdf,agronomy,0.67,454c95229128,unverified
2,S03,National-Agriculture-Production-Report-2025.pdf,statistics,7.58,72a8a2f73d12,unverified
3,S04,Olwande_Smallholder_Maize_Efficiency_Kenya.pdf,economics,0.47,46d25e01d128,unverified
4,S05,Planting_strategies_of_maize_farmers_in_Kenya_...,agronomy,1.08,af7816c7cbfc,unverified
5,S06,CIMMYT_Maize_Diseases_Field_Identification_4th...,disease,2.24,c8bcadc375c6,free to share (CIMMYT Intl Public Good)
6,S07,CIMMYT_MLN_Technical_Manual_Disease_Management...,disease,6.94,7511f8694c1c,CIMMYT publication
7,S08,KALRO_MLND_Kenya_Training_Manual.pdf,disease,1.96,0f77e5f1e843,Government of Kenya / KALRO
8,S09,KEPHIS_National_Crop_Variety_List_2025.pdf,seed,3.68,4c11714e9a90,KEPHIS / Government of Kenya
9,S10,MoA_Sustainable_Agricultural_Livelihood_Restor...,agronomy,37.37,f2e65123f305,Government of Kenya


24 PDFs inventoried, all unique, ready to extract.


### 2.2 PDF text extraction

Text is extracted **page by page** and page numbers are kept - they are the citation unit all the way to the chat answer. PyMuPDF does not OCR: a scanned page comes back empty and is flagged as a *review candidate* in the audit. Layout such as tables degrades to running text; check important numbers against the original PDF.

In [88]:
def extract_pages(sources):
    pages = []
    for src in sources:
        path = PDF_DIR / src["filename"]
        doc = pymupdf.open(path)
        if doc.needs_pass:
            raise ValueError(f"{src['filename']} is encrypted - decrypt it first.")
        for page in doc:
            text = page.get_text("text") or ""
            pages.append({
                "source_id": src["source_id"],
                "filename": src["filename"],
                "title": src["title"],
                "topic": src["topic"],
                "page": page.number + 1,
                "n_words": len(text.split()),
                "text": text,
            })
        doc.close()
    return pages

t0 = time.perf_counter()
pdf_pages = extract_pages(PDF_SOURCES)
elapsed = time.perf_counter() - t0

audit = (
    pd.DataFrame(pdf_pages)
    .groupby(["source_id", "filename"], as_index=False)
    .agg(pages=("page", "count"), words=("n_words", "sum"),
         short_pages=("n_words", lambda s: int((s < 50).sum())))
)
display(audit)

shortest = pd.DataFrame(pdf_pages).nsmallest(8, "n_words")[["source_id", "page", "n_words"]]
print("Review candidates (thinnest pages - check they are not failed extractions):")
display(shortest)
print(f"extracted {len(pdf_pages)} pages / {sum(p['n_words'] for p in pdf_pages):,} words in {elapsed:.1f}s")

,source_id,filename,pages,words,short_pages
0,S01,diversity in maize production environments and...,36,1519,17
1,S02,MAIZE GROWERS GUIDE.pdf,19,3385,1
2,S03,National-Agriculture-Production-Report-2025.pdf,202,41719,56
3,S04,Olwande_Smallholder_Maize_Efficiency_Kenya.pdf,28,1743,11
4,S05,Planting_strategies_of_maize_farmers_in_Kenya_...,13,8372,0
5,S06,CIMMYT_Maize_Diseases_Field_Identification_4th...,124,8614,58
6,S07,CIMMYT_MLN_Technical_Manual_Disease_Management...,80,31732,13
7,S08,KALRO_MLND_Kenya_Training_Manual.pdf,48,9223,7
8,S09,KEPHIS_National_Crop_Variety_List_2025.pdf,232,56950,0
9,S10,MoA_Sustainable_Agricultural_Livelihood_Restor...,297,126317,7


Review candidates (thinnest pages - check they are not failed extractions):


,source_id,page,n_words
29,S01,30,0
30,S01,31,0
31,S01,32,0
32,S01,33,0
33,S01,34,0
34,S01,35,0
35,S01,36,0
298,S06,1,0


extracted 1770 pages / 507,823 words in 3.7s


### 2.3 Text cleaning

- Unicode normalised to NFC, null bytes and non-breaking spaces removed.
- Hyphenated line breaks rejoined (`inter-\nnational` -> `international`).
- All whitespace collapsed to single spaces - chunking works on word windows, so line and paragraph breaks carry no information downstream.

Cleaning is deliberately conservative: no stopword removal, no stemming, no lowercasing - that is the embedding model's job.

In [89]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", " ").replace("\xa0", " ")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[\uE000-\uF8FF]", " ", text)
    return text.strip()

before = sum(len(p["text"]) for p in pdf_pages)
for page in pdf_pages:
    page["text"] = clean_text(page["text"])
    page["n_words"] = len(page["text"].split())
after = sum(len(p["text"]) for p in pdf_pages)

empty_pages = [p for p in pdf_pages if not p["text"]]
print(f"cleaned {before:,} -> {after:,} chars ({before - after:,} removed)")
print(f"empty pages after cleaning: {len(empty_pages)}")
if empty_pages:
    display(pd.DataFrame(empty_pages)[["source_id", "page", "n_words"]])

cleaned 3,446,964 -> 3,286,305 chars (160,659 removed)
empty pages after cleaning: 28


,source_id,page,n_words
0,S01,30,0
1,S01,31,0
2,S01,32,0
3,S01,33,0
4,S01,34,0
5,S01,35,0
6,S01,36,0
7,S06,1,0
8,S07,2,0
9,S07,79,0


### 2.4 Language tagging
- This counts common English and Swahili words to label each page en, sw or unknown.
- A stopword-ratio heuristic tags each page `en`, `sw` or `unknown`. It is deliberately simple and **not** a language detector of record - it only feeds corpus statistics and the routing sanity checks in section 4. `unknown` pages are treated as English (the corpus is predominantly English); `langdetect` or `fasttext lid.176` can replace this cell without changing downstream code.

In [90]:
EN_STOP = {
    "the", "and", "of", "to", "in", "a", "is", "that", "for", "on", "with", "as", "by",
    "at", "from", "this", "are", "be", "it", "or", "an", "was", "were", "which", "has",
    "have", "not", "but", "their", "its", "will", "can", "one", "than", "also", "into",
}
SW_STOP = {
    "ya", "na", "wa", "kwa", "cha", "hii", "hiyo", "hizo", "wao", "yake", "wake",
    "katika", "kuhusu", "ambayo", "ambao", "hivyo", "lakini", "sana", "aidha", "bali",
    "kabla", "baada", "hadi", "pia", "kila", "wakulima", "shamba", "kilimo", "mahindi",
    "mazao", "udongo", "mbolea", "wadudu", "ugonjwa", "magonjwa", "mbegu", "nchini",
    "kutumia", "kupitia", "zaidi", "chini", "juu", "wakati", "hali", "hewa",
    # question words and common farmer-question terms
    "ni", "nini", "gani", "yapi", "ipi", "nani", "wapi", "lini", "vipi", "je",
    "jinsi", "naweza", "nifanye", "ninawezaje", "ninapaswa",
    "kupanda", "kuvuna", "dalili", "tiba", "kutibu", "kuzuia",
}

def detect_language(text):
    tokens = [t for t in re.findall(r"[a-zA-Z]+", text.lower()) if len(t) > 1]
    if not tokens:
        return "unknown", 0, 0
    en_hits = sum(t in EN_STOP for t in tokens)
    sw_hits = sum(t in SW_STOP for t in tokens)
    if sw_hits >= 5 and sw_hits > en_hits:
        return "sw", sw_hits, en_hits
    if en_hits >= 5:
        return "en", en_hits, sw_hits
    return "unknown", en_hits, sw_hits

for page in pdf_pages:
    page["language"], page["en_hits"], page["sw_hits"] = detect_language(page["text"])

page_lang = pd.DataFrame(pdf_pages).groupby(["source_id", "language"], as_index=False).agg(pages=("page", "count"))
display(page_lang.pivot(index="source_id", columns="language", values="pages").fillna(0).astype(int))

for src in PDF_SOURCES:
    pages = [p for p in pdf_pages if p["source_id"] == src["source_id"]]
    votes = [p["language"] for p in pages]
    doc_lang = max(set(votes), key=votes.count)
    src["language"] = "en" if doc_lang == "unknown" else doc_lang
    print(f"{src['source_id']}  {doc_lang:7s} (majority of {len(pages)} pages)  {src['title'][:50]}")

language,en,unknown
source_id,,
S01,9,27
S02,17,2
S03,111,91
S04,23,5
S05,13,0
S06,67,57
S07,70,10
S08,43,5
S09,227,5


S01  unknown (majority of 36 pages)  Diversity in maize production environments and pra
S02  en      (majority of 19 pages)  Maize Growers Guide
S03  en      (majority of 202 pages)  National Agriculture Production Report 2025
S04  en      (majority of 28 pages)  Smallholder maize efficiency in Kenya (Olwande)
S05  en      (majority of 13 pages)  Planting strategies of maize farmers in Kenya
S06  en      (majority of 124 pages)  Maize Diseases: A Guide for Field Identification (
S07  en      (majority of 80 pages)  Maize Lethal Necrosis (MLN): A Technical Manual fo
S08  en      (majority of 48 pages)  Understanding & Managing Maize Lethal Necrosis Dis
S09  en      (majority of 232 pages)  National Crop Variety List - Kenya (2025 edition)
S10  en      (majority of 297 pages)  Sustainable Agricultural Livelihood Restoration Tr
S11  en      (majority of 14 pages)  Overview of soil fertility & soil-testing programs
S12  en      (majority of 1 pages)  Kenya Central Fertilizer Use Optimizer 

### 2.5 Seed conversion (`DISEASE_INFO`)

Each disease becomes one compact document combining pathogen, symptoms, favourable conditions and management advice. Seed pages use `source_id` `S00` and `page` 0 (no PDF page exists), so citations read *"DISEASE_INFO"* instead of a page number.

In [91]:
seed_pages = []
for cls, info in DISEASE_INFO.items():
    parts = [f"Maize leaf disease: {cls.replace('_', ' ')}. Pathogen: {info['pathogen']}."]
    if info["pathogen"] != "-":
        parts.append(f"Symptoms: {info['symptoms']}")
    parts.append(f"Favourable conditions: {info['favourable']}")
    parts.append(f"Management: {info['management']}")
    seed_pages.append({
        "source_id": "S00",
        "filename": None,
        "title": f"Maize leaf disease: {cls}",
        "topic": "disease",
        "page": 0,
        "n_words": len(" ".join(parts).split()),
        "text": clean_text(" ".join(parts)),
        "language": "en",
        "disease": cls,
    })

print(f"{len(seed_pages)} seed documents created:")
for page in seed_pages:
    print(f"  {page['title']:35s} {page['n_words']:3d} words")

4 seed documents created:
  Maize leaf disease: Blight          108 words
  Maize leaf disease: Common_Rust      97 words
  Maize leaf disease: Gray_Leaf_Spot  103 words
  Maize leaf disease: Healthy          43 words


### 2.6 Chunking

This splits pages into overlapping 220-word pieces, never across pages, and drops duplicates and tiny leftovers.

In [92]:
def make_chunks(pages, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS, min_words=MIN_CHUNK_WORDS):
    if chunk_words < 1 or not 0 <= overlap_words < chunk_words:
        raise ValueError("Require chunk_words > 1 and 0 <= overlap_words < chunk_words.")
    step = chunk_words - overlap_words
    chunks, duplicates, seen = [], [], {}
    for page in pages:
        words = page["text"].split()
        if not words:
            continue
        prev_end, window_index = 0, 0
        for start in range(0, len(words), step):
            window = words[start:start + chunk_words]
            if start + len(window) <= prev_end:
                continue    
            if len(window) < min_words and start > 0 and len(words) > min_words:
                continue   
            prev_end = start + len(window)
            window_index += 1
            text = " ".join(window)
            text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
            if page.get("disease"):
                chunk_id = f"{page['source_id']}-p{page['page']:04d}-{page['disease'].lower()}"
            else:
                chunk_id = f"{page['source_id']}-p{page['page']:04d}-c{window_index:03d}"
            if text_hash in seen:
                duplicates.append({"removed": chunk_id, "kept": seen[text_hash]})
                continue
            seen[text_hash] = chunk_id
            chunks.append({
                "chunk_id": chunk_id,
                "source_id": page["source_id"],
                "filename": page.get("filename"),
                "title": page["title"],
                "topic": page["topic"],
                "language": page["language"],
                "page": page["page"],
                "disease": page.get("disease"),
                "n_words": len(window),
                "text_sha256": text_hash,
                "text": text,
            })
    return chunks, duplicates

all_pages = seed_pages + pdf_pages
chunks, duplicate_chunks = make_chunks(all_pages)
print(f"{len(all_pages)} pages -> {len(chunks)} chunks "
      f"({len(duplicate_chunks)} exact duplicates removed)")

1774 pages -> 3447 chunks (1 exact duplicates removed)


### 2.7 Chunk quality inspection

This shows chunk counts, length stats and sample text so you can spot problems.

In [93]:
frame = pd.DataFrame(chunks)
stats = frame.groupby(["source_id", "title"], as_index=False).agg(
    chunks=("chunk_id", "count"), pages=("page", "nunique"),
    words=("n_words", "sum"), mean_words=("n_words", "mean"),
)
stats["mean_words"] = stats["mean_words"].round(0).astype(int)
display(stats)

lengths = frame["n_words"]
print(f"chunk length: min={lengths.min()}  median={int(lengths.median())}  max={lengths.max()}")
print(f"chunks below {MIN_CHUNK_WORDS} words: {(lengths < MIN_CHUNK_WORDS).sum()}")
print("language distribution:", frame["language"].value_counts().to_dict())
print("topic distribution   :", frame["topic"].value_counts().to_dict())

print("\nFive shortest chunks (review their usefulness):")
display(frame.nsmallest(5, "n_words")[["chunk_id", "topic", "n_words", "page"]])

print("\nSample seed chunk (disease):")
disease_chunk = frame[frame["source_id"] == "S00"].iloc[0]
print(f"[{disease_chunk['chunk_id']}] {disease_chunk['text'][:400]}...")

print("\nSample PDF chunk:")
pdf_chunk = frame[frame["source_id"] != "S00"].iloc[0]
print(f"[{pdf_chunk['chunk_id']}] {pdf_chunk['text'][:400]}...")

,source_id,title,chunks,pages,words,mean_words
0,S00,Maize leaf disease: Blight,1,1,108,108
1,S00,Maize leaf disease: Common_Rust,1,1,97,97
2,S00,Maize leaf disease: Gray_Leaf_Spot,1,1,103,103
3,S00,Maize leaf disease: Healthy,1,1,43,43
4,S01,Diversity in maize production environments and...,29,29,1434,49
5,S02,Maize Growers Guide,24,19,3547,148
6,S03,National Agriculture Production Report 2025,326,202,46415,142
7,S04,Smallholder maize efficiency in Kenya (Olwande),27,27,1732,64
8,S05,Planting strategies of maize farmers in Kenya,48,13,9772,204
9,S06,Maize Diseases: A Guide for Field Identificati...,126,123,8723,69


chunk length: min=1  median=217  max=220
chunks below 30 words: 161
language distribution: {'en': 3201, 'unknown': 246}
topic distribution   : {'agronomy': 888, 'policy': 661, 'disease': 407, 'postharvest': 393, 'seed': 372, 'pest': 350, 'statistics': 326, 'economics': 27, 'fertilizer': 23}

Five shortest chunks (review their usefulness):


,chunk_id,topic,n_words,page
860,S08-p0048-c001,disease,1,48
1247,S10-p0010-c001,agronomy,1,10
1250,S10-p0012-c001,agronomy,1,12
32,S01-p0029-c001,agronomy,2,29
479,S06-p0021-c001,disease,2,21



Sample seed chunk (disease):
[S00-p0000-blight] Maize leaf disease: Blight. Pathogen: Exserohilum turcicum (formerly Bipolaris turcica). Symptoms: Long, cigar-shaped grey-green to tan lesions running parallel to the veins. Lesions start on lower leaves and move upward. Under favourable conditions they coalesce and the whole leaf dies, giving the crop a burnt appearance. Favourable conditions: Cool to moderate temperatures (18-27 C) with high hu...

Sample PDF chunk:
[S01-p0001-c001] Diversity in Maize Production Environments and Practices in Kenya Introduction and Conference Overview Mary W. K. Mathenge Tegemeo Institute – Egerton University 1...


### 2.8 Persist

Four files land in `rag_data/`:

| File | Contents |
|---|---|
| `chunks.jsonl` | one chunk per line - the unit that gets embedded in section 3 |
| `pages.jsonl` | cleaned page texts - lets section 7 re-chunk with new settings without re-extracting the PDFs |
| `source_manifest.json` | provenance: hashes, page/word counts, languages, config used |
| `page_audit.csv` | one row per page: words kept and an `empty` flag for scanned pages that need review |

A round-trip read verifies the files are valid before moving on.

In [94]:
DATA_DIR.mkdir(exist_ok=True)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

chunks_path = DATA_DIR / "chunks.jsonl"
pages_path = DATA_DIR / "pages.jsonl"
manifest_path = DATA_DIR / "source_manifest.json"
audit_path = DATA_DIR / "page_audit.csv"

write_jsonl(chunks_path, chunks)
write_jsonl(pages_path, all_pages)

manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "config": {
        "embedding_model": EMBEDDING_MODEL,
        "chunk_words": CHUNK_WORDS,
        "overlap_words": OVERLAP_WORDS,
        "languages": list(LANGUAGES),
        "seed": SEED,
    },
    "sources": [
        {
            "source_id": src["source_id"],
            "filename": src["filename"],
            "title": src["title"],
            "topic": src["topic"],
            "language": src["language"],
            "licence": src["licence"],
            "sha256": SOURCE_HASHES.get(src["source_id"]),
            "chunks": sum(1 for c in chunks if c["source_id"] == src["source_id"]),
        }
        for src in SOURCES
    ],
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(pdf_pages).drop(columns=["text"]).to_csv(audit_path, index=False)

# round-trip verification
with open(chunks_path, encoding="utf-8") as handle:
    reloaded = [json.loads(line) for line in handle]
assert len(reloaded) == len(chunks), "chunks.jsonl round-trip failed"
assert reloaded[0]["chunk_id"] == chunks[0]["chunk_id"]
assert len(list(open(pages_path, encoding="utf-8"))) == len(all_pages)

for path in (chunks_path, pages_path, manifest_path, audit_path):
    print(f"{path.relative_to(ROOT)}  {path.stat().st_size / 1024:.1f} KB")
print(f"\n{len(chunks)} chunks persisted - section 3 embeds them.")

rag_data/chunks.jsonl  4898.0 KB
rag_data/pages.jsonl  3693.3 KB
rag_data/source_manifest.json  9.3 KB
rag_data/page_audit.csv  245.3 KB

3447 chunks persisted - section 3 embeds them.


## 3. Embeddings & Vector Store

An **embedding** is just a list of numbers - 384 of them - that captures *meaning*. Texts with similar meanings end up close together, even in different languages.

```
question  --> "query: Ni mbolea gani ya mahindi?"    --> [0.12, -0.45, ...] --.
                                                                      closest --> retrieved chunks
each chunk --> "passage: Fertiliser rates are..."    --> [0.11, -0.47, ...] --'
```

- **Model:** `intfloat/multilingual-e5-small` - small (~120 MB), free, runs locally, trained on many languages **including Swahili**. None of our PDFs are Swahili, so this is what lets a Swahili question find English text (proven in 3.5).
- **Prefixes:** e5 expects `query: ` before questions and `passage: ` before documents. Same text with a different prefix gives a different vector, so never mix them up.
- **Normalisation:** every vector is scaled to length 1, so similarity is a plain dot product (cosine similarity).

### 3.1 Load the embedding model

This loads **multilingual-e5-small**, which turns text into numbers that capture meaning, including Swahili.

In [95]:
from sentence_transformers import SentenceTransformer

t0 = time.perf_counter()
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
print(f"loaded {EMBEDDING_MODEL} on {DEVICE} in {time.perf_counter() - t0:.1f}s")
print(f"max sequence length: {embedder.max_seq_length} tokens  (chunks are ~220 words, safely below)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Exception ignored while calling deallocator <function tqdm.__del__ at 0x11b0c1c70>:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/notebook.py", line 271, in close
    if self.disable:
AttributeError: 'tqdm' object has no attribute 'disable'
Exception ignored while calling deallocator <function tqdm.__del__ at 0x11b0c1c70>:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/notebook.py", line 271, in close
    if self.disable:
AttributeError: 'tqdm' object has no attribute 'disable'


loaded intfloat/multilingual-e5-small on mps in 15.1s
max sequence length: 512 tokens  (chunks are ~220 words, safely below)


### 3.2 Embed the corpus

This converts every chunk to a 384-number vector, adding the passage: prefix the model expects.

In [96]:
def embed_texts(texts, kind):
    """kind is "query" (questions) or "passage" (documents). e5 needs the prefix."""
    prefix = "query: " if kind == "query" else "passage: "
    vectors = embedder.encode(
        [prefix + text for text in texts],
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return np.asarray(vectors, dtype="float32")

t0 = time.perf_counter()
chunk_vectors = embed_texts([chunk["text"] for chunk in chunks], "passage")
print(f"embedded {chunk_vectors.shape[0]} chunks -> {chunk_vectors.shape[1]} dimensions "
      f"in {time.perf_counter() - t0:.1f}s")

Batches:   0%|          | 0/108 [00:00<?, ?it/s]

embedded 3447 chunks -> 384 dimensions in 78.0s


### 3.3 Build the vector index

This stores the vectors in a searchable index and saves it to chunks.npy.

In [97]:
class NumpyIndex:
    def __init__(self, vectors):
        self.vectors = np.asarray(vectors, dtype="float32")

    def search(self, queries, k):
        scores = np.asarray(queries, dtype="float32") @ self.vectors.T
        order = np.argsort(-scores, axis=1)[:, :k]
        return np.take_along_axis(scores, order, 1), order


index = NumpyIndex(chunk_vectors)
index_path = ARTIFACT_DIR / "chunks.npy"
np.save(index_path, chunk_vectors)
print(f"index holds {index.vectors.shape[0]} vectors, saved to {index_path.relative_to(ROOT)}")

index holds 3447 vectors, saved to rag_artifacts/chunks.npy


### 3.4 Index statistics

This checks vector count, dimensions and normalisation.

In [98]:
norms = np.linalg.norm(chunk_vectors, axis=1)
print(f"model           : {EMBEDDING_MODEL}")
print(f"matrix          : {chunk_vectors.shape[0]} vectors x {chunk_vectors.shape[1]} dims")
print(f"index type      : {type(index).__name__} (exact search)")
print(f"file size       : {index_path.stat().st_size / 1024:.1f} KB")
print(f"vector norms    : min {norms.min():.4f}  max {norms.max():.4f}  (all ~1.0 = normalised)")
print(f"chunk row order : matches chunks.jsonl line order")

model           : intfloat/multilingual-e5-small
matrix          : 3447 vectors x 384 dims
index type      : NumpyIndex (exact search)
file size       : 5170.6 KB
vector norms    : min 1.0000  max 1.0000  (all ~1.0 = normalised)
chunk row order : matches chunks.jsonl line order


### 3.5 Cross-lingual sanity check
This defines search(), which embeds a question and returns the closest chunks. It's tested with 2 English and 2 Swahili questions. Read the results two ways:

- **Scores** — a Swahili question scoring in the same ballpark as English (~0.75+) proves the two languages sit close together in this vector space. That is the signal everything rests on.
- **Rankings** — English rankings must be obviously correct. Swahili rankings are allowed to be weaker *at this stage*: section 4 translates Swahili questions into English **before** retrieval (the translation pivot), and section 7 tunes against measured metrics. This check only asks: does the cross-lingual signal exist at all?

Watch for one pattern: pure number-table chunks from the statistics report score surprisingly well on anything — section 2.7 exists to help you decide whether to drop them from the corpus.

In [99]:
def search(query, k=TOP_K):
    """Return [(score, chunk), ...] for the k chunks closest to the query."""
    query_vector = embed_texts([query], "query")
    scores, row_ids = index.search(query_vector, k)
    return [(float(score), chunks[row]) for score, row in zip(scores[0], row_ids[0])]


TEST_QUERIES = [
    "What fertiliser should I apply to maize?",
    "What are the symptoms of common rust?",
    "Ni mbolea gani inafaa kwa mahindi?",   
    "Magonjwa ya mahindi ni yapi?", 
]

for question in TEST_QUERIES:
    print(f"\nQ: {question}")
    for score, chunk in search(question, 3):
        snippet = chunk["text"][:90].replace("\n", " ")
        print(f"   {score:.3f}  [{chunk['chunk_id']}] {chunk['title'][:40]:42s} {snippet}...")


Q: What fertiliser should I apply to maize?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   0.896  [S02-p0002-c001] Maize Growers Guide                        2 Soils and climate Maize grows best on deep, well-drained, fertile soils, and where total...
   0.881  [S02-p0004-c001] Maize Growers Guide                        4 The basal fertiliser must be applied before or at the time of planting, preferably cuppe...
   0.876  [S02-p0017-c001] Maize Growers Guide                        17 Mid-season management 1. Weed control, especially in the first 10 weeks after crop emer...

Q: What are the symptoms of common rust?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   0.870  [S00-p0000-common_rust] Maize leaf disease: Common_Rust            Maize leaf disease: Common Rust. Pathogen: Puccinia sorghi. Symptoms: Small, oval-to-elong...
   0.869  [S06-p0014-c001] Maize Diseases: A Guide for Field Identi   9 Maize rusts The three leaf rusts on maize are common rust, polysora rust, and tropical r...
   0.847  [S08-p0022-c001] Understanding & Managing Maize Lethal Ne   22 Common rust (Puccinia sorgi) The disease is characterized by elongate raised bumps (pus...

Q: Ni mbolea gani inafaa kwa mahindi?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   0.810  [S03-p0189-c002] National Agriculture Production Report 2   10 14 Kilifi 403 76 391 112 372 42 243 45 26 6 Kirinyaga 93 108 86 71 73 25 110 149 40 61 ...
   0.807  [S09-p0131-c002] National Crop Variety List - Kenya (2025   AEZ: LM 14 Sites: Mwea, Kitui, Homabay, Bura,Kisu mu,Hola and Alupe 5-6 2.2 -2.4 Medium ma...
   0.807  [S03-p0115-c002] National Agriculture Production Report 2   1,255.1 323.9 78.8 34.0 110.0 37.3 29.7 3,859,793.7 1,738,320.0 6,263,852.0 2,388,594.3 2,...

Q: Magonjwa ya mahindi ni yapi?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   0.791  [S07-p0077-c002] Maize Lethal Necrosis (MLN): A Technical   maize seed value chain in Africa. Governments • Mandate and enforce synchronized maize pla...
   0.791  [S07-p0007-c001] Maize Lethal Necrosis (MLN): A Technical   iii Technical Manual for MLN Disease Management Table of Contents S. No. Chapters Authors ...
   0.791  [S10-p0167-c003] Sustainable Agricultural Livelihood Rest   (g) 32.7 - - Chick weight at 8 weeks (g) 438.9 - - Body weight at first egg (g) 1630.0 - -...


## 4. Retrieval & Augmentation

### 4.1 BM25 keyword search
- This section finds chunks by exact word matches.
- Section 3 searched by *meaning*. BM25 searches by *exact words* - dumb but reliable. For a question like "DAP fertiliser rates" it beats embeddings, because the acronym only appears verbatim in one place. It also cannot match across languages, which is why we translate first.

In [100]:
from rank_bm25 import BM25Okapi

bm25_tokens = [re.findall(r"[a-z0-9']+", chunk["text"].lower()) for chunk in chunks]
bm25 = BM25Okapi(bm25_tokens)

def bm25_search(query, k=TOP_K):
    """Return [(score, chunk), ...] for the k chunks with the best keyword overlap."""
    scores = np.asarray(bm25.get_scores(re.findall(r"[a-z0-9']+", query.lower())))
    order = scores.argsort()[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in order if scores[i] > 0]

for score, chunk in bm25_search("DAP fertiliser application rates", 3):
    print(f"  {score:7.2f}  [{chunk['chunk_id']}] {chunk['title'][:40]:42s} {chunk['text'][:70]}...")

    16.86  [S12-p0001-c001] Kenya Central Fertilizer Use Optimizer (   Kenya Central Fertilizer Use Optimizer: paper version, (CAN-DAP) Nov. ...
    15.98  [S10-p0038-c002] Sustainable Agricultural Livelihood Rest   Fertilisers Inorganic nitrogen fertilisers come in many different form...
    15.98  [S13-p0001-c001] Kenya Lower Western Fertilizer Use Optim   Kenya Lower Western Fertilizer Use Optimizer: paper version, July 2023...


### 4.2 Hybrid search (rank fusion)

This combines the meaning search and the keyword search by rank (reciprocal rank fusion), because their scores can't be compared directly.

In [101]:
def hybrid_search(query, k=TOP_K):
    """Dense + BM25 fused by rank. Returns [(rrf_score, chunk), ...], best first."""
    dense = search(query, k * 3)
    sparse = bm25_search(query, k * 3)

    votes = {}
    for rank, (_, chunk) in enumerate(dense, start=1):
        votes[chunk["chunk_id"]] = votes.get(chunk["chunk_id"], 0) + 1 / (60 + rank)
    for rank, (_, chunk) in enumerate(sparse, start=1):
        votes[chunk["chunk_id"]] = votes.get(chunk["chunk_id"], 0) + 1 / (60 + rank)

    ranked = sorted(votes, key=votes.get, reverse=True)[:k]
    by_id = {chunk["chunk_id"]: chunk for chunk in chunks}
    return [(votes[chunk_id], by_id[chunk_id]) for chunk_id in ranked]

for score, chunk in hybrid_search("DAP fertiliser application rates", 3):
    print(f"  {score:.4f}  [{chunk['chunk_id']}] {chunk['title'][:40]:42s} {chunk['text'][:70]}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  0.0325  [S12-p0001-c001] Kenya Central Fertilizer Use Optimizer (   Kenya Central Fertilizer Use Optimizer: paper version, (CAN-DAP) Nov. ...
  0.0323  [S13-p0001-c001] Kenya Lower Western Fertilizer Use Optim   Kenya Lower Western Fertilizer Use Optimizer: paper version, July 2023...
  0.0308  [S13-p0001-c003] Kenya Lower Western Fertilizer Use Optim   DAP (1 BB for 6 m). Side dress by broadcasting 48 kg urea at (7-14 DAE...


### 4.3 Language routing (the Swahili pivot)

The corpus is English, so a Swahili question is **translated to English first**, searched, and translated back later (section 8). Two small pieces:

1. `looks_swahili()` - the stopword list from 2.4, with a lower bar (2 words instead of 5) because questions are short.
2. `translate_to_english()` - `facebook/nllb-200-distilled-600M`, a free model running locally that handles 200 languages including Swahili (`swh_Latn`).

In [102]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

TRANS_MODEL = "facebook/nllb-200-distilled-600M"
trans_tokenizer = AutoTokenizer.from_pretrained(TRANS_MODEL, src_lang="swh_Latn", tgt_lang="eng_Latn")
trans_model = AutoModelForSeq2SeqLM.from_pretrained(TRANS_MODEL).to(DEVICE)

def looks_swahili(text):
    words = set(re.findall(r"[a-zA-Z]+", text.lower()))
    return len(words & SW_STOP) >= 2

def translate_to_english(text):
    tokens = trans_tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    tokens = tokens.to(trans_model.device)
    output = trans_model.generate(**tokens, max_length=200,
                                   forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("eng_Latn"))
    english = trans_tokenizer.decode(output[0], skip_special_tokens=True)
    return re.sub(r"\bcorn\b", "maize", english, flags=re.IGNORECASE)

print(f"loaded {TRANS_MODEL} on {DEVICE}")
for test in ["Ni mbolea gani inafaa kwa mahindi?", "Magonjwa ya mahindi ni yapi?", "What fertiliser should I use?"]:
    lang = "sw" if looks_swahili(test) else "en"
    routed = translate_to_english(test) if lang == "sw" else test
    print(f"  [{lang}] {test}")
    print(f"       -> {routed}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

loaded facebook/nllb-200-distilled-600M on mps
  [sw] Ni mbolea gani inafaa kwa mahindi?
       -> What kind of fertilizer is suitable for maize?
  [sw] Magonjwa ya mahindi ni yapi?
       -> What are maize diseases?
  [en] What fertiliser should I use?
       -> What fertiliser should I use?


### 4.4 `retrieve()` - the whole chain in one function
- one function that detects the language, translates if needed, and runs the hybrid search to return the top 5 chunks.
- `retrieve(question)` = route -> translate if needed -> hybrid search -> top-5 chunks. Everything later (sections 5-8) calls only this function, so tuning in section 7 never ripples outward.

In [103]:
def retrieve(query, k=TOP_K):
    """Return (english_query, [(score, chunk), ...])."""
    routed = translate_to_english(query) if looks_swahili(query) else query
    return routed, hybrid_search(routed, k)

for question in TEST_QUERIES:
    routed, hits = retrieve(question)
    print(f"\nQ  : {question}")
    if routed != question:
        print(f"EN : {routed}")
    for score, chunk in hits[:3]:
        snippet = chunk["text"][:75].replace("\n", " ")
        print(f"   {score:.4f}  [{chunk['chunk_id']:28s}] {chunk['title'][:38]:40s} {snippet}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q  : What fertiliser should I apply to maize?
   0.0303  [S10-p0039-c001              ] Sustainable Agricultural Livelihood Re   22 Banding Fertiliser is placed in a continuous band at the bottom of the f...
   0.0302  [S10-p0038-c003              ] Sustainable Agricultural Livelihood Re   chloride. Potassium nitrate is easy to apply, because it does not pull mois...
   0.0293  [S10-p0038-c002              ] Sustainable Agricultural Livelihood Re   Fertilisers Inorganic nitrogen fertilisers come in many different forms, su...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q  : What are the symptoms of common rust?
   0.0325  [S06-p0014-c001              ] Maize Diseases: A Guide for Field Iden   9 Maize rusts The three leaf rusts on maize are common rust, polysora rust,...
   0.0317  [S08-p0022-c001              ] Understanding & Managing Maize Lethal    22 Common rust (Puccinia sorgi) The disease is characterized by elongate ra...
   0.0315  [S00-p0000-common_rust       ] Maize leaf disease: Common_Rust          Maize leaf disease: Common Rust. Pathogen: Puccinia sorghi. Symptoms: Small...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q  : Ni mbolea gani inafaa kwa mahindi?
EN : What kind of fertilizer is suitable for maize?
   0.0164  [S02-p0002-c001              ] Maize Growers Guide                      2 Soils and climate Maize grows best on deep, well-drained, fertile soils, ...
   0.0164  [S19-p0036-c001              ] Maize Post-harvest Operations Compendi   MAIZE:Post-harvest Operations Page 35 Fig. 12. The "Matraca"planter with a ...
   0.0161  [S04-p0019-c001              ] Smallholder maize efficiency in Kenya    1. More widespread and intensive use of modern farming technologies (cont) ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q  : Magonjwa ya mahindi ni yapi?
EN : What are maize diseases?
   0.0297  [S06-p0006-c002              ] Maize Diseases: A Guide for Field Iden   for maize diseases: fungi, bacteria, viruses, and mollicutes. Fungal diseas...
   0.0292  [S06-p0006-c001              ] Maize Diseases: A Guide for Field Iden   1 Introduction This popular booklet, already in its fourth edition, is desi...
   0.0164  [S08-p0029-c001              ] Understanding & Managing Maize Lethal    29 a b a. Irregular distribution of the normal green color causing a mosaic...


### 4.5 Augmentation - building the prompt
Here we paste the 5 chunks into a template with these rules for the language model:

In [104]:
PROMPT_TEMPLATE = """You are the Kenya Maize Advisory Chatbot, an assistant for smallholder maize farmers.

Answer ONLY from the numbered sources below.
- Cite sources like [1] or [2][3] after every claim.
- If the sources do not contain the answer, say so and advise consulting a county extension officer.
- Reply in the SAME language the question was asked in (English or Kiswahili).
- For pesticides, only state what the source says (product, rate, waiting period) and add a wear-gloves/wash warning.
- End every answer with: "For decisions that affect your harvest, consult your county extension officer."

Sources:
{context}

Question: {question}
Answer:"""


def build_prompt(question, hits):
    context = "\n\n".join(
        f"[{i}] {chunk['title']} ({chunk['source_id']}, page {chunk['page']}):\n{chunk['text']}"
        for i, (_, chunk) in enumerate(hits, start=1)
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)


sample_question = "What are the symptoms of common rust and how do I manage it?"
_, sample_hits = retrieve(sample_question)
sample_prompt = build_prompt(sample_question, sample_hits)
print(f"prompt length: {len(sample_prompt)} chars, {len(sample_hits)} sources cited")
print("-" * 70)
print(sample_prompt[:1200])
print("...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

prompt length: 5966 chars, 5 sources cited
----------------------------------------------------------------------
You are the Kenya Maize Advisory Chatbot, an assistant for smallholder maize farmers.

Answer ONLY from the numbered sources below.
- Cite sources like [1] or [2][3] after every claim.
- If the sources do not contain the answer, say so and advise consulting a county extension officer.
- Reply in the SAME language the question was asked in (English or Kiswahili).
- For pesticides, only state what the source says (product, rate, waiting period) and add a wear-gloves/wash warning.
- End every answer with: "For decisions that affect your harvest, consult your county extension officer."

Sources:
[1] Maize Diseases: A Guide for Field Identification (4th ed., CIMMYT) (S06, page 14):
9 Maize rusts The three leaf rusts on maize are common rust, polysora rust, and tropical rust. Common rust Puccinia sorghi The disease is found worldwide in subtropical, temperate, and highland enviro

## 5. Evaluation

We check that retrieval actually works before building anything on top of it.

Twelve questions with known correct pages are asked four ways:

- **dense** - search by meaning only
- **bm25** - search by keywords only
- **hybrid** - both fused together (section 4.2)
- **hybrid without translation** - what happens if the Swahili pivot (4.3) is skipped

Two scores, both from 0 to 1, higher is better:

- **Hit@5** - how often a correct page shows up in the top 5 results
- **MRR** - how high the first correct page ranks (1.0 = correct page ranked first)

### 5.1 The test questions

- Builds EVAL_SET: 12 test questions (8 English, 4 Swahili).
- Stores the answer key: each question lists the chunk IDs that contain the correct answer, such as S02-p0004-c001.
- Checks the labels: it confirms every chunk ID in the answer key really exists. It prints "unknown chunk ids" if any are wrong.

In [105]:
EVAL_SET = [
    {"q": "What are the symptoms of common rust?", "lang": "en",
     "relevant": ["S00-p0000-common_rust"]},
    {"q": "How do I manage grey leaf spot?", "lang": "en",
     "relevant": ["S00-p0000-gray_leaf_spot"]},
    {"q": "When should I plant maize in Kenya?", "lang": "en",
     "relevant": ["S02-p0016-c001", "S02-p0016-c002", "S02-p0012-c002"]},
    {"q": "How much basal fertiliser should I apply at planting?", "lang": "en",
     "relevant": ["S02-p0004-c001", "S02-p0003-c001"]},
    {"q": "Which maize varieties are popular in Kenya?", "lang": "en",
     "relevant": ["S01-p0019-c001", "S01-p0011-c001"]},
    {"q": "How much rainfall does maize need?", "lang": "en",
     "relevant": ["S02-p0002-c001"]},
    {"q": "How do I harvest maize to reduce losses?", "lang": "en",
     "relevant": ["S02-p0018-c001"]},
    {"q": "How much maize was produced per county in 2024?", "lang": "en",
     "relevant": ["S03-p0186-c001", "S03-p0187-c001"]},
    {"q": "Ni mbolea gani inafaa kwa mahindi?", "lang": "sw",
     "relevant": ["S02-p0003-c001", "S02-p0004-c001"]},
    {"q": "Magonjwa ya mahindi ni yapi?", "lang": "sw",
     "relevant": ["S00-p0000-common_rust", "S00-p0000-blight",
                  "S00-p0000-gray_leaf_spot"]},
    {"q": "Ni lini wakati wa kupanda mahindi?", "lang": "sw",
     "relevant": ["S02-p0016-c001", "S02-p0016-c002"]},
    {"q": "Jinsi ya kuvuna mahindi bila hasara?", "lang": "sw",
     "relevant": ["S02-p0018-c001"]},
]

known_ids = {chunk["chunk_id"] for chunk in chunks}
unknown = [cid for item in EVAL_SET for cid in item["relevant"] if cid not in known_ids]
print(f"{len(EVAL_SET)} questions "
      f"({sum(1 for i in EVAL_SET if i['lang'] == 'en')} English, "
      f"{sum(1 for i in EVAL_SET if i['lang'] == 'sw')} Swahili)")
print("unknown chunk ids:", unknown if unknown else "none - all labels point at real chunks")

12 questions (8 English, 4 Swahili)
unknown chunk ids: none - all labels point at real chunks


### 5.2 Scoring

- Defines score_system(): it takes any search function and runs all 12 questions through it.
- For each question: it finds the position of the first correct chunk in the top 5.
- Returns two scores:
  - hit@5 is the share of questions with a correct chunk in the top 5.
  - mrr rewards ranking the correct chunk higher. First place scores 1, second scores 0.5, third scores 0.33, and a miss scores 0.

In [106]:
def score_system(system_fn):
    hits, reciprocal_ranks = 0, []
    for item in EVAL_SET:
        retrieved = system_fn(item)
        rank = next((i + 1 for i, cid in enumerate(retrieved) if cid in item["relevant"]), None)
        hits += rank is not None
        reciprocal_ranks.append(1 / rank if rank else 0.0)
    return {"hit@5": hits / len(EVAL_SET),
            "mrr": sum(reciprocal_ranks) / len(EVAL_SET)}

### 5.3 Four searches compared

First the Swahili questions are translated once, so the top three systems all search the same English text and only the search method differs. The fourth system searches the raw question, which is the fair way to measure what the pivot contributes.

In [107]:
def routed_query(item):
    if item["lang"] == "sw" and looks_swahili(item["q"]):
        return translate_to_english(item["q"])
    return item["q"]

ROUTED = {}
for item in EVAL_SET:
    ROUTED[item["q"]] = routed_query(item)
    if ROUTED[item["q"]] != item["q"]:
        print(f"  [sw] {item['q']}")
        print(f"       -> {ROUTED[item['q']]}")

def get_dense(item):
    return [chunk["chunk_id"] for _, chunk in search(ROUTED[item["q"]])]

def get_bm25(item):
    return [chunk["chunk_id"] for _, chunk in bm25_search(ROUTED[item["q"]])]

def get_hybrid(item):
    return [chunk["chunk_id"] for _, chunk in hybrid_search(ROUTED[item["q"]])]

def get_no_pivot(item):
    return [chunk["chunk_id"] for _, chunk in hybrid_search(item["q"])]

systems = {"dense": get_dense, "bm25": get_bm25, "hybrid": get_hybrid,
           "hybrid without translation": get_no_pivot}
results = {name: score_system(fn) for name, fn in systems.items()}
print()
print(pd.DataFrame(results).T.to_string(float_format=lambda x: f"{x:.3f}"))

  [sw] Ni mbolea gani inafaa kwa mahindi?
       -> What kind of fertilizer is suitable for maize?
  [sw] Magonjwa ya mahindi ni yapi?
       -> What are maize diseases?
  [sw] Ni lini wakati wa kupanda mahindi?
       -> When is the time to plant maize?
  [sw] Jinsi ya kuvuna mahindi bila hasara?
       -> How to harvest maize without loss?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


                            hit@5   mrr
dense                       0.667 0.472
bm25                        0.167 0.104
hybrid                      0.500 0.207
hybrid without translation  0.250 0.128


### 5.4 What did we learn?

- Picks the winner: the system with the highest mrr.
- Lists the misses: questions where the hybrid search found no correct chunk in the top 5.
- Saves a report: the results go to rag_artifacts/retrieval_eval.json, with the question count, the scores, the best system and a timestamp.

In [108]:
best = max(results, key=lambda name: results[name]["mrr"])
print(f"best system: {best}  (hit@5 {results[best]['hit@5']:.3f}, mrr {results[best]['mrr']:.3f})")

print("\nquestions the hybrid system missed:")
missed = [item for item in EVAL_SET
          if not any(cid in item["relevant"] for cid in get_hybrid(item))]
for item in missed:
    print(f"  [{item['lang']}] {item['q']}")
if not missed:
    print("  none - every question found a correct page in the top 5")

report = {"n_questions": len(EVAL_SET), "results": results,
          "best_system": best, "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
out_path = ARTIFACT_DIR / "retrieval_eval.json"
out_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"\nsaved to {out_path.relative_to(ROOT)}")

best system: dense  (hit@5 0.667, mrr 0.472)

questions the hybrid system missed:


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  [en] How do I manage grey leaf spot?
  [en] When should I plant maize in Kenya?
  [en] Which maize varieties are popular in Kenya?
  [en] How much rainfall does maize need?
  [en] How much maize was produced per county in 2024?
  [sw] Ni lini wakati wa kupanda mahindi?

saved to rag_artifacts/retrieval_eval.json


## 6. Guardrails & Safety

Guardrails are checks the code runs itself - the model promising to behave does not count.

Three pieces:

- **6.1** refuses the wrong question *before* any searching happens
- **6.2** shows why the similarity score cannot do the refusing (measured, not assumed)
- **6.3** inspects the finished answer *before* anyone sees it

Section 8 chains them: refuse -> retrieve -> generate -> validate -> retry or show.

### 6.1 Refusing the wrong questions

Every question first passes `refuse()`. Three rules, in order:

1. **Live prices and subsidies** are never guessed - the sources are static documents, so the refusal points to real-world places to check instead.
2. **Human health** is never advised on - this is a maize chatbot.
3. **Off-topic** - if the question contains not one maize or farming word, it is off-topic.

Each refusal is a fixed sentence in the question's own language. `DISCLAIMER` is the same sentence the prompt template in 4.5 requires at the end of every answer, defined here once so validation can check it.

In [109]:
DISCLAIMER = "For decisions that affect your harvest, consult your county extension officer."

TOPIC_WORDS = [
    "maize", "corn", "crop", "farm", "field", "soil", "rain", "seed", "fertilis",
    "fertiliz", "manure", "pest", "disease", "insect", "weed", "harvest", "yield",
    "plant", "variety", "varieties", "irrigation", "storage", "grain", "cob",
    "leaf", "blight", "rust", "hectare", "acre", "agronom", "extension",
    "mahindi", "mbolea", "shamba", "kilimo", "wakulima", "mazao", "ugonjwa",
    "magonjwa", "wadudu", "mbegu", "udongo", "mvua", "kuvuna", "mavuno",
    "kupanda", "dalili", "tiba", "kuzuia", "ipil",
]

REFUSAL_RULES = [
    ("live_prices", [r"\bprices?\b", r"how much is", r"\bbei\b", r"ruzuku",
                     r"subsid", r"market rate"]),
    ("human_medicine", [r"malaria", r"\bhuman\b", r"binadamu", r"\bdawa\b",
                        r"medicine"]),
]

REFUSALS = {
    "live_prices": {
        "en": "I do not have live market prices or subsidy information - my sources are "
              "static documents. Check today's prices with your local produce buyer, the "
              "county market office, or the e-Mkulima platform.",
        "sw": "Sina bei za soko za sasa wala taarifa za ruzuku - vyanzo vyangu ni vitabu "
              "vya zamani. Uliza mnunuzi wako wa mazao, ofisi ya soko ya kaunti, au tumia "
              "jukwaa la e-Mkulima.",
    },
    "human_medicine": {
        "en": "I only advise on growing maize. Questions about human health should go to "
              "a clinic or hospital.",
        "sw": "Nashauri tu kuhusu ukuzaji wa mahindi. Maswali kuhusu afya ya binadamu "
              "yawaoneshe kwa daktari.",
    },
    "off_topic": {
        "en": "I only answer questions about growing maize in Kenya - diseases, planting, "
              "fertiliser, pests, harvest and storage. Ask me about maize!",
        "sw": "Najibu maswali tu kuhusu ukuzaji wa mahindi nchini Kenya - magonjwa, "
              "kupanda, mbolea, wadudu, mavuno na uhifadhi. Niulize kuhusu mahindi!",
    },
}


def refuse(question):
    text = question.lower()
    lang = "sw" if looks_swahili(question) else "en"
    for category, patterns in REFUSAL_RULES:
        if any(re.search(pattern, text) for pattern in patterns):
            return REFUSALS[category][lang]
    if not re.search(r"\b(" + "|".join(TOPIC_WORDS) + ")", text):
        return REFUSALS["off_topic"][lang]
    return None

### 6.2 Why the similarity score is not a gate

A tempting shortcut would be: refuse whatever scores low. The measurement below shows why that fails - junk questions score just as high as real ones, because the model finds *some* maize page for almost anything. So the word rules above do the refusing, not the score. Section 7 may revisit this with better metrics.

In [110]:
SCORE_EXAMPLES = [
    ("real", "When should I plant maize?"),
    ("real", "What are the symptoms of common rust?"),
    ("real", "Jinsi ya kuvuna mahindi bila hasara?"),
    ("junk", "How do I make Italian pasta?"),
    ("junk", "Who won the football world cup?"),
    ("junk", "Tell me about Bitcoin mining"),
    ("junk", "How much is a bag of maize today?"),
]

vectors = embedder.encode(["query: " + q for _, q in SCORE_EXAMPLES],
                          normalize_embeddings=True, convert_to_numpy=True).astype("float32")
scores, ids = index.search(vectors, 1)
for (kind, q), score in zip(SCORE_EXAMPLES, scores[:, 0]):
    print(f"  {kind:4s} {float(score):.3f}  {q}")
print("\nreal questions and junk questions land in the same 0.75-0.89 range,")
print("so the score cannot gate anything. refuse() in 6.1 does that job instead.")

  real 0.885  When should I plant maize?
  real 0.870  What are the symptoms of common rust?
  real 0.830  Jinsi ya kuvuna mahindi bila hasara?
  junk 0.822  How do I make Italian pasta?
  junk 0.772  Who won the football world cup?
  junk 0.827  Tell me about Bitcoin mining
  junk 0.868  How much is a bag of maize today?

real questions and junk questions land in the same 0.75-0.89 range,
so the score cannot gate anything. refuse() in 6.1 does that job instead.


### 6.3 Checking the finished answer

After the model answers, `validate_answer()` checks five things:

1. it carries at least one citation like `[1]`, and none beyond the sources it was given
2. it ends with the disclaimer
3. it answers in the question's own language
4. it does not promise guaranteed results
5. pesticide advice only appears together with an application rate

In [111]:
def validate_answer(answer, question, n_sources):
    problems = []
    citations = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    if not citations:
        problems.append("no citation")
    if any(n > n_sources for n in citations):
        problems.append("citation points outside the given sources")
    if not answer.strip().endswith(DISCLAIMER):
        problems.append("disclaimer missing at the end")
    if looks_swahili(question) != looks_swahili(answer):
        problems.append("answer language does not match the question")
    if re.search(r"\b(guaranteed|guarantee|guarantees|hakikisha)\b|\b100%", answer, re.I):
        problems.append("promises certainty")
    if re.search(r"\b(pesticide|insecticide|fungicide)s?\b", answer, re.I) \
            and not re.search(r"\d+\s*(ml|g|kg|%|litre|liter)\b", answer, re.I):
        problems.append("pesticide advice without an application rate")
    return problems

### 6.4 Testing the guardrails

Five questions that must be refused, two that must pass, and eight answers with (or without) each problem. The section is only done when this cell prints zero failures.

In [112]:
fails = 0

print("input rules")
refuse_cases = [
    ("How much is a bag of maize today?", True),
    ("Bei ya mahindi leo ni ngapi?", True),
    ("Who won the football world cup?", True),
    ("Soka ni mchezo gani?", True),
    ("Can maize plants cure malaria in humans?", True),
    ("When should I plant maize?", False),
    ("Jinsi ya kupanda mahindi?", False),
]
for question, should_refuse in refuse_cases:
    got = refuse(question) is not None
    ok = got == should_refuse
    fails += not ok
    print(f"  {'PASS' if ok else 'FAIL'}  refused={got}  {question}")

print("\noutput validator")
answer_cases = [
    ("clean English answer",
     "Common rust shows cinnamon-brown pustules [1]. Use resistant hybrids [1]. " + DISCLAIMER,
     "What are the symptoms of common rust?", []),
    ("clean Swahili answer",
     "Tumia mbegu za aina ya kukabiliana na ugonjwa [1]. Ondoa magugu ya mahindi [1]. " + DISCLAIMER,
     "Ni mbolea gani inafaa kwa mahindi?", []),
    ("no citation",
     "The pustules are brown and powdery. " + DISCLAIMER,
     "What are the symptoms of common rust?", ["no citation"]),
    ("no disclaimer",
     "The pustules are brown and powdery [1].",
     "What are the symptoms of common rust?", ["disclaimer"]),
    ("pesticide without rate",
     "Spray the insecticide when you see pests [1]. " + DISCLAIMER,
     "How do I control stem borer?", ["rate"]),
    ("promises certainty",
     "This will 100% cure the disease [1]. " + DISCLAIMER,
     "How do I manage rust?", ["certainty"]),
    ("wrong language",
     "Kupanda mahindi kwa wakati ni muhimu [1]. " + DISCLAIMER,
     "When should I plant maize?", ["language"]),
    ("citation out of range",
     "The answer is here [7]. " + DISCLAIMER,
     "What is common rust?", ["outside"]),
]
for label, answer, question, expected in answer_cases:
    problems = validate_answer(answer, question, n_sources=1)
    if expected:
        ok = all(any(sub in p for p in problems) for sub in expected) and \
             all(problems) and len(problems) == len(expected)
    else:
        ok = not problems
    fails += not ok
    print(f"  {'PASS' if ok else 'FAIL'}  {label:25s} problems={problems}")

print(f"\n{fails} failures")

input rules
  PASS  refused=True  How much is a bag of maize today?
  PASS  refused=True  Bei ya mahindi leo ni ngapi?
  PASS  refused=True  Who won the football world cup?
  PASS  refused=True  Soka ni mchezo gani?
  PASS  refused=True  Can maize plants cure malaria in humans?
  PASS  refused=False  When should I plant maize?
  PASS  refused=False  Jinsi ya kupanda mahindi?

output validator
  PASS  clean English answer      problems=[]
  PASS  clean Swahili answer      problems=[]
  PASS  no citation               problems=['no citation']
  PASS  no disclaimer             problems=['disclaimer missing at the end']
  PASS  pesticide without rate    problems=['pesticide advice without an application rate']
  PASS  promises certainty        problems=['promises certainty']
  PASS  wrong language            problems=['answer language does not match the question']
  PASS  citation out of range     problems=['citation points outside the given sources']

0 failures


## 7. Optimization

### 7.1 Why the hybrid search lagged

The section 5 results showed the hybrid search doing worse than the plain meaning search. This section tunes how the two searches are combined until the hybrid earns its place.

Three settings decide how much each half is trusted:

- `dense_weight`
- `depth`
- `rrf_k` 

### 7.2 Making the fusion tunable

This merges the meaning-search ranking and the keyword-search ranking into one top 5.

In [113]:
def fuse_ranks(dense_ranks, sparse_ranks, k=TOP_K, dense_weight=0.5, depth=10, rrf_k=10):
    votes = {}
    for rank, chunk_id in enumerate(dense_ranks[:k * depth], start=1):
        votes[chunk_id] = votes.get(chunk_id, 0) + dense_weight / (rrf_k + rank)
    for rank, chunk_id in enumerate(sparse_ranks[:k * depth], start=1):
        votes[chunk_id] = votes.get(chunk_id, 0) + (1 - dense_weight) / (rrf_k + rank)
    ranked = sorted(votes, key=votes.get, reverse=True)[:k]
    return [(chunk_id, votes[chunk_id]) for chunk_id in ranked]

### 7.3 Sweeping the settings

Runs both searches once per question: it stores the long result lists (up to 50 chunks each) in **candidate_ranks**.

In [114]:
MAX_DEPTH = 10
candidate_ranks = {}
for item in EVAL_SET:
    query = ROUTED[item["q"]]
    candidate_ranks[item["q"]] = {
        "dense": [chunk["chunk_id"] for _, chunk in search(query, TOP_K * MAX_DEPTH)],
        "sparse": [chunk["chunk_id"] for _, chunk in bm25_search(query, TOP_K * MAX_DEPTH)],
    }


def score_config(dense_weight, depth, rrf_k):
    hits, reciprocal_ranks = 0, []
    for item in EVAL_SET:
        ranks = candidate_ranks[item["q"]]
        ranked = [cid for cid, _ in fuse_ranks(ranks["dense"], ranks["sparse"],
                                               dense_weight=dense_weight, depth=depth, rrf_k=rrf_k)]
        rank = next((i + 1 for i, cid in enumerate(ranked) if cid in item["relevant"]), None)
        hits += rank is not None
        reciprocal_ranks.append(1 / rank if rank else 0.0)
    return {"hit@5": hits / len(EVAL_SET), "mrr": sum(reciprocal_ranks) / len(EVAL_SET)}


sweep = []
for dense_weight in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    for depth in (1, 3, 5, 10):
        for rrf_k in (10, 30, 60):
            sweep.append({"dense_weight": dense_weight, "depth": depth, "rrf_k": rrf_k,
                          **score_config(dense_weight, depth, rrf_k)})

best = max(sweep, key=lambda row: (row["hit@5"], row["mrr"]))
print(f"{len(sweep)} settings tested - best: dense_weight={best['dense_weight']}, "
      f"depth={best['depth']}, rrf_k={best['rrf_k']}")
print("\ntop 8 settings:")
print(pd.DataFrame(sweep).sort_values(["hit@5", "mrr"], ascending=False).head(8)
      .to_string(index=False, float_format=lambda value: f"{value:.3f}"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

84 settings tested - best: dense_weight=0.9, depth=10, rrf_k=30

top 8 settings:
 dense_weight  depth  rrf_k  hit@5   mrr
        0.900     10     30  0.750 0.422
        0.700     10     10  0.750 0.360
        0.800      3     10  0.667 0.431
        0.900      3     10  0.667 0.431
        0.900      5     10  0.667 0.431
        0.900     10     10  0.667 0.431
        0.800      5     10  0.667 0.424
        0.800     10     10  0.667 0.424


### 7.4 Winner

The best setting replaces the fusion used everywhere from here on, including `retrieve()`. The same questions are scored again so the change is visible next to the section 5 numbers.

In [115]:
BEST_CONFIG = {"dense_weight": best["dense_weight"], "depth": best["depth"], "rrf_k": best["rrf_k"]}


def hybrid_search(query, k=TOP_K):
    dense_ranks = [chunk["chunk_id"] for _, chunk in search(query, k * BEST_CONFIG["depth"])]
    sparse_ranks = [chunk["chunk_id"] for _, chunk in bm25_search(query, k * BEST_CONFIG["depth"])]
    by_id = {chunk["chunk_id"]: chunk for chunk in chunks}
    return [(score, by_id[chunk_id]) for chunk_id, score in
            fuse_ranks(dense_ranks, sparse_ranks, k=k, **BEST_CONFIG)]


before = results["hybrid"]
after = score_system(get_hybrid)
print(f"hybrid fusion settings: {BEST_CONFIG}\n")
print(pd.DataFrame({"before (4.2)": before, "after (7.4)": after})
      .to_string(float_format=lambda value: f"{value:.3f}"))
print(f"\ndense alone for reference: hit@5 {results['dense']['hit@5']:.3f}, mrr {results['dense']['mrr']:.3f}")

optimization_report = {"best_config": BEST_CONFIG, "hybrid_before": before, "hybrid_after": after,
                       "dense_reference": results["dense"], "n_questions": len(EVAL_SET),
                       "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
out_path = ARTIFACT_DIR / "optimization.json"
out_path.write_text(json.dumps(optimization_report, indent=2), encoding="utf-8")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

hybrid fusion settings: {'dense_weight': 0.9, 'depth': 10, 'rrf_k': 30}

       before (4.2)  after (7.4)
hit@5         0.500        0.750
mrr           0.207        0.422

dense alone for reference: hit@5 0.667, mrr 0.472


389

## 8. Inference Pipeline

This section wires the pieces together into one function: refuse, retrieve, generate, validate, and retry once when a rule is broken.

### 8.1 Loading the generator

Qwen2.5-1.5B-Instruct runs in this notebook through transformers. It turns the prompt from 4.5 into an answer, in English.

In [116]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float16).to(DEVICE)
gen_model.eval()

print(f"loaded {GEN_MODEL} on {DEVICE}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-1.5B-Instruct on mps


### 8.2 Turning a prompt into an answer

The prompt is wrapped in Qwen's chat format and generated greedily, so the same question gives the same answer. Only the new tokens are decoded, the prompt is cut off the front.

In [117]:
def generate(prompt, max_new_tokens=400):
    messages = [{'role': 'user', 'content': prompt}]
    text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(text, return_tensors="pt").to(gen_model.device)
    output = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True)

### 8.3 Speaking Swahili back

The generator answers in English. For a Swahili question the body is translated back with the same NLLB model from 4.3, running in the other direction. The English `DISCLAIMER` is put back on the end, because 6.3 checks for that exact sentence.

In [118]:
sw_tokenizer = AutoTokenizer.from_pretrained(TRANS_MODEL, src_lang="eng_Latn", tgt_lang="swh_Latn")

def translate_to_swahili(text):
    tokens = sw_tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(trans_model.device)
    output = trans_model.generate(**tokens, max_length=512,
                                  forced_bos_token_id=sw_tokenizer.convert_tokens_to_ids("swh_Latn"))
    return sw_tokenizer.decode(output[0], skip_special_tokens=True)

### 8.4 Chaining things together

`ask()` is the pipeline in one place. A refused question stops immediately. Otherwise retrieve, prompt, generate, validate. If validation finds a problem the prompt gets one nudge and the answer is generated once more.

In [119]:
def ask(question, k=TOP_K):
    refusal = refuse(question)
    if refusal:
        return {"question": question, "answer": refusal, "sources": [],
                "refused": True, "problems": []}

    english_query, hits = retrieve(question, k)
    prompt = build_prompt(english_query, hits)
    answer = generate(prompt)
    problems = validate_answer(answer, question, len(hits))

    if problems:
        retry_prompt = prompt + "\n\nCite every claim with [n] and end with the exact disclaimer sentence."
        answer = generate(retry_prompt)
        problems = validate_answer(answer, question, len(hits))

    if looks_swahili(question) and not looks_swahili(answer):
        body = answer.split(DISCLAIMER)[0].strip()
        answer = translate_to_swahili(body).strip() + " " + DISCLAIMER
        problems = validate_answer(answer, question, len(hits))

    return {"question": question, "answer": answer, "sources": [c["chunk_id"] for _, c in hits],
            "refused": False, "problems": problems}

### 8.5 Testing the pipeline

In [120]:
questions = [
    "What are the symptoms of common rust and how do I manage it?",
    "How much basal fertiliser should I apply at planting?",
    "Magonjwa ya mahindi ni yapi?",
    "How much is a bag of maize today?",
]
for question in questions:
    result = ask(question)
    print("=" * 70)
    print(f"Q: {result['question']}")
    print("REFUSED" if result["refused"] else f"A: {result['answer']}")
    print(f"sources: {result['sources']}")
    print(f"problems: {result['problems']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q: What are the symptoms of common rust and how do I manage it?
A: Common rust, also known as Puccinia sorghi, causes small, elongate, powdery pustules on both leaf surfaces. These pustules are initially dark brown but darken further as the plant matures. The disease is more noticeable when plants are approaching tasseling. To manage common rust, use resistant maize varieties whenever possible because they are the least expensive method. Additionally, remove any volunteer maize and other alternate hosts like oxalis species. Fungicides are generally not economically viable unless a resistant variety cannot be used due to early arrival of the disease. Consult your county extension officer for specific recommendations based on local conditions. For decisions that affect your harvest, consult your county extension officer.
sources: ['S06-p0014-c001', 'S00-p0000-common_rust', 'S08-p0022-c002', 'S08-p0022-c001', 'S10-p0094-c002']
problems: ['no citation', 'pesticide advice without an applica

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q: How much basal fertiliser should I apply at planting?
A: According to the Maize Growers Guide (S02, page 4), the basal fertiliser must be applied before or at the time of planting, preferably cupped into the planting hole, or applied in a band below and to the side of the seed with the mechanical planter. The top dressing should be applied when the maize at 3.5 to 6 Weeks After crop Emergence-WACE. Top dressing splits may be recommended in sandy to sandy loam soils. A maximum of 3 usually is ideal. Basal fertilisers containing zinc are recommended for application every two to three years on sandy soils. If boronated fertilisers are not normally applied as part of the rotation, then a boronated fertiliser should be applied every three years. 

The approximate application rates (to the nearest half-bag) of inorganic fertilisers to crops can be determined from the following tables:

Basal fertilizers e.g. 7,14,7

Plant Spacing Cup size Between Between 5 8 12 16 22 Rows Plants Fertilize

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q: Magonjwa ya mahindi ni yapi?
A: Magonjwa ya mahindi yameainishwa chini ya kuvu, bakteria, virusi, na molekuli. Magonjwa haya hujulikana kwa dalili zao ambazo hutofautiana kulingana na hatua ya maendeleo ya mahindi. Magonjwa maalum yaliyotajwa ni pamoja na ugonjwa wa majani ya mahindi ya kaskazini, ugonjwa wa ugonjwa wa mazao ya mahindi, na doa la majani ya kijivu. Kila ugonjwa una sifa za kipekee na inahitaji mikakati maalum ya usimamizi. For decisions that affect your harvest, consult your county extension officer.
sources: ['S06-p0006-c002', 'S08-p0020-c001', 'S00-p0000-blight', 'S06-p0006-c001', 'S00-p0000-gray_leaf_spot']
problems: ['no citation']
Q: How much is a bag of maize today?
REFUSED
sources: []
problems: []


In [140]:
q1 = "What fertilizers are the best for maize in Kenya?"
q2 = "Ni mbolea gani bora kwa mahindi nchini Kenya?"
q3 = "How old is Lionel Messi?"

ans1 = ask(q1)
ans2 = ask(q2)
ans3 = ask(q3)

print(ans1["answer"])
print(ans2["answer"])
print(ans3["answer"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Based on the information provided, the best fertilizers for maize in Kenya are those containing potassium (K) and possibly nitrogen (N). Specifically, the study mentions that adding potassium at 40 kg K2O/ha in Bungoma and Trans Nzoia to maize and 80 kg K2O in rice in Mwea showed a clear response. Additionally, the report notes that the Government has allocated funds to NCPB to undertake these functions on Agency basis, indicating that NCPB markets seed maize for Kenya. However, without specific details on the types of fertilizers used, I cannot provide a definitive list of the best fertilizers. For decisions that affect your harvest, consult your county extension officer.
Mbolea bora kwa mahindi nchini Kenya inategemea mambo mbalimbali kama vile aina ya udongo, viwango vya virutubisho, na mahitaji maalum ya mazao. Kwa ujumla, potasiamu (K) ni ya manufaa kwa kuboresha mavuno ya mahindi. Kulingana na utafiti juu ya urekebishaji wa Bodi ya Taifa ya Mazao na Mazao (NCPB), kuongeza 40 kg K

## 9. Export

The app should not re-run this notebook. This section writes everything inference needs into one folder, with checksums, so it can be loaded and verified later.

### 9.1 Writing the export

One folder holds the corpus, the vectors and the settings actually used. A `manifest.json` records a sha256 for each file, so a later loader can tell if anything was edited or copied wrong.

In [121]:
EXPORT_DIR = ARTIFACT_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)

export_config = {
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GEN_MODEL,
    "translation_model": TRANS_MODEL,
    "languages": list(LANGUAGES),
    "chunk_words": CHUNK_WORDS,
    "overlap_words": OVERLAP_WORDS,
    "min_chunk_words": MIN_CHUNK_WORDS,
    "top_k": TOP_K,
    "hybrid": BEST_CONFIG,
    "chunk_count": len(chunks),
    "embedding_dim": int(chunk_vectors.shape[1]),
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(EXPORT_DIR / "config.json").write_text(json.dumps(export_config, indent=2), encoding="utf-8")
write_jsonl(EXPORT_DIR / "chunks.jsonl", chunks)
np.save(EXPORT_DIR / "vectors.npy", chunk_vectors)


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


EXPORT_FILES = ("chunks.jsonl", "vectors.npy", "config.json")
manifest = {name: {"sha256": sha256_of(EXPORT_DIR / name),
                   "bytes": (EXPORT_DIR / name).stat().st_size} for name in EXPORT_FILES}
(EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

for name in EXPORT_FILES:
    print(f"{name:16s} {manifest[name]['bytes'] / 1024:8.1f} KB  {manifest[name]['sha256'][:12]}")
print(f"\n{len(chunks)} chunks and {chunk_vectors.shape} vectors exported to {EXPORT_DIR.relative_to(ROOT)}")

chunks.jsonl       4898.0 KB  41b40fe84fc9
vectors.npy        5170.6 KB  3fb5d6e830b8
config.json           0.5 KB  d56928c95469

3447 chunks and (3447, 384) vectors exported to rag_artifacts/export


### 9.2 Loading it back

The loader checks every file against the manifest first, then rebuilds the corpus, the vectors and the settings. A changed or corrupt file stops it with an error instead of quietly returning bad results.

In [122]:
def load_export(export_dir):
    export_dir = Path(export_dir)
    manifest = json.loads((export_dir / "manifest.json").read_text(encoding="utf-8"))
    for name, info in manifest.items():
        if sha256_of(export_dir / name) != info["sha256"]:
            raise ValueError(f"export file changed or corrupt: {name}")
    config = json.loads((export_dir / "config.json").read_text(encoding="utf-8"))
    corpus = [json.loads(line) for line in
              (export_dir / "chunks.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
    vectors = np.load(export_dir / "vectors.npy")
    return {"config": config, "chunks": corpus, "vectors": vectors}


loaded = load_export(EXPORT_DIR)
print(f"loaded {len(loaded['chunks'])} chunks and {loaded['vectors'].shape} vectors")
print(f"embedding model : {loaded['config']['embedding_model']}")
print(f"generation model: {loaded['config']['generation_model']}")
print(f"hybrid settings : {loaded['config']['hybrid']}")

loaded 3447 chunks and (3447, 384) vectors
embedding model : intfloat/multilingual-e5-small
generation model: Qwen/Qwen2.5-1.5B-Instruct
hybrid settings : {'dense_weight': 0.9, 'depth': 10, 'rrf_k': 30}


## 10. Database & Chat History

The Streamlit app saves farmer profiles and conversations. This section defines the MySQL tables and the small set of functions the app calls. Nothing connects until a function runs, so the notebook executes with no database present.

### 10.1 Connection

Details come from environment variables, never from notebook text. `db_connection()` opens one connection per call, commits on success and rolls back on error.

In [135]:
import contextlib
import os
import mysql.connector

DB_ENV_KEYS = ("MYSQL_HOST", "MYSQL_DATABASE", "MYSQL_USER", "MYSQL_PASSWORD")


@contextlib.contextmanager
def db_connection():
    if mysql is None:
        raise RuntimeError("mysql-connector-python is not installed")
    missing = [key for key in DB_ENV_KEYS if not os.environ.get(key)]
    if missing:
        raise ValueError("Missing environment variables: " + ", ".join(missing))
    connection = mysql.connector.connect(
        host=os.environ["MYSQL_HOST"],
        port=int(os.environ.get("MYSQL_PORT", "3306")),
        database=os.environ["MYSQL_DATABASE"],
        user=os.environ["MYSQL_USER"],
        password=os.environ["MYSQL_PASSWORD"],
        charset="utf8mb4",
        autocommit=False,
    )
    try:
        yield connection
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()

### 10.2 Schema

Three tables matching the README: `farmers`, `chat_sessions` and `chat_messages`. Creation is off by default and runs only against the configured development database.

In [136]:
DATABASE_SQL = [
    """CREATE TABLE IF NOT EXISTS farmers (
        id INT AUTO_INCREMENT PRIMARY KEY,
        full_name VARCHAR(120) NOT NULL,
        county VARCHAR(60),
        preferred_lang CHAR(2) NOT NULL DEFAULT 'en',
        created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
    """CREATE TABLE IF NOT EXISTS chat_sessions (
        id INT AUTO_INCREMENT PRIMARY KEY,
        farmer_id INT NOT NULL,
        started_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
        language CHAR(2) NOT NULL DEFAULT 'en',
        FOREIGN KEY (farmer_id) REFERENCES farmers(id) ON DELETE CASCADE
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
    """CREATE TABLE IF NOT EXISTS chat_messages (
        id INT AUTO_INCREMENT PRIMARY KEY,
        session_id INT NOT NULL,
        farmer_id INT NOT NULL,
        role VARCHAR(10) NOT NULL,
        message MEDIUMTEXT NOT NULL,
        language CHAR(2) NOT NULL DEFAULT 'en',
        sources JSON,
        feedback TINYINT NULL,
        created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (session_id) REFERENCES chat_sessions(id) ON DELETE CASCADE,
        FOREIGN KEY (farmer_id) REFERENCES farmers(id) ON DELETE CASCADE
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
]


def create_schema():
    with db_connection() as connection:
        cursor = connection.cursor()
        for statement in DATABASE_SQL:
            cursor.execute(statement)
        cursor.close()


CREATE_SCHEMA = False
if CREATE_SCHEMA:
    create_schema()
    print("schema ready")

### 10.3 Saving a conversation

A farmer is one row in `farmers`, a conversation is one row in `chat_sessions`. Each `ask()` result is stored as two rows: the question with role `user`, the answer with role `assistant`.

In [137]:
def create_farmer(full_name, county=None, preferred_lang="en"):
    if not full_name or len(full_name) > 120:
        raise ValueError("full_name must be 1-120 characters")
    if preferred_lang not in ("en", "sw"):
        raise ValueError("preferred_lang must be en or sw")
    with db_connection() as connection:
        cursor = connection.cursor()
        cursor.execute("INSERT INTO farmers (full_name, county, preferred_lang) VALUES (%s, %s, %s)",
                       (full_name, county, preferred_lang))
        farmer_id = cursor.lastrowid
        cursor.close()
    return farmer_id


def start_session(farmer_id, language="en"):
    with db_connection() as connection:
        cursor = connection.cursor()
        cursor.execute("INSERT INTO chat_sessions (farmer_id, language) VALUES (%s, %s)",
                       (farmer_id, language))
        session_id = cursor.lastrowid
        cursor.close()
    return session_id


def store_message(farmer_id, session_id, question, result):
    language = "sw" if looks_swahili(question) else "en"
    rows = [
        (session_id, farmer_id, "user", question, language, None),
        (session_id, farmer_id, "assistant", result["answer"], language,
         json.dumps(result.get("sources", []))),
    ]
    with db_connection() as connection:
        cursor = connection.cursor()
        cursor.executemany(
            "INSERT INTO chat_messages (session_id, farmer_id, role, message, language, sources) "
            "VALUES (%s, %s, %s, %s, %s, %s)", rows)
        cursor.close()

### 10.4 History and feedback

`read_history()` returns a conversation oldest first. `set_feedback()` records one thumbs up or down on an answer row.

In [138]:
def read_history(farmer_id, session_id, limit=100):
    with db_connection() as connection:
        cursor = connection.cursor(dictionary=True)
        cursor.execute(
            "SELECT id, role, message, language, sources, feedback, created_at "
            "FROM chat_messages WHERE farmer_id = %s AND session_id = %s "
            "ORDER BY id ASC LIMIT %s", (farmer_id, session_id, limit))
        rows = cursor.fetchall()
        cursor.close()
    return rows


def set_feedback(message_id, helpful):
    if not isinstance(helpful, bool):
        raise ValueError("helpful must be True or False")
    with db_connection() as connection:
        cursor = connection.cursor()
        cursor.execute("UPDATE chat_messages SET feedback = %s WHERE id = %s",
                       (1 if helpful else -1, message_id))
        cursor.close()

### 10.5 Smoke test

Runs only against a development database when `RUN_DATABASE_SMOKE_TESTS` is True. It creates a synthetic farmer, stores a question and an answer, reads them back, records feedback and deletes the farmer again. The cascade removes the session and messages with it.

In [139]:
RUN_DATABASE_SMOKE_TESTS = False

if RUN_DATABASE_SMOKE_TESTS:
    demo_result = {"answer": "Synthetic test answer, not agronomy advice.",
                   "sources": ["S00-p0000-common_rust"]}
    farmer = create_farmer("Notebook test farmer", county="Nairobi")
    session = start_session(farmer)
    store_message(farmer, session, "What are the symptoms of common rust?", demo_result)
    history = read_history(farmer, session)
    set_feedback(history[-1]["id"], helpful=True)
    for row in read_history(farmer, session):
        print(f"[{row['role']:9s}] {row['message'][:55]}  feedback={row['feedback']}")
    with db_connection() as connection:
        cursor = connection.cursor()
        cursor.execute("DELETE FROM farmers WHERE id = %s", (farmer,))
        cursor.close()
    print("smoke test passed and the synthetic farmer was removed")